# Tuning kelompok destinasi — eksplorasi terdaftar

**Tujuan:** memilih konfigurasi menggunakan validation kelompok; evaluasi independen menunggu data baru.

Dataset 27 destinasi yang pernah digunakan pada eksperimen sebelumnya diperlakukan seluruhnya sebagai **development**. Membagi ulang data yang sama tidak menjadikannya independen kembali. Tidak ada nilai yang disebut final test accuracy di notebook ini.

Alur: input → tetapkan protokol/fold → LDA train-only → embedding train-only → kandidat BiLSTM → ringkasan validation → konfigurasi terpilih → evaluasi eksternal ditunda.

Kelas satu destinasi selalu train-only; tabel cakupan mengungkap kelas yang tidak bisa dievaluasi. Tiga fold dipakai berulang untuk memilih K, checkpoint, dan hiperparameter, sehingga skor validation adalah diagnostik seleksi yang optimistis, bukan estimasi tanpa bias. [Protokol](../docs/TUNING_PROTOCOL.md).

## 00 · Setup

Input: repository dan mode. Proses: instalasi. Output: modul, folder run, hash. Gunakan runtime Colab baru. MODE full menjalankan pencarian yang ditetapkan; smoke hanya memeriksa alur.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, sys, os, hashlib, json

MODE = "full"  # "full" atau "smoke"
RUN_DEEP = True
SEED = 42
DATA_PATH = None  # None: dataset Madura otomatis, tanpa upload.
OUTPUT_DIR = Path("tuning_kelompok_madura") / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
BRANCH = "feat/streamlit-wisata"
REPOSITORY = "https://github.com/MalikHariyanto/crispy_recom_machine_system.git"
if MODE not in {"full", "smoke"}:
    raise ValueError("MODE harus full atau smoke")
checkout = Path.cwd() / "wisata_madura_tuning"
repo_root = next((p for p in (Path.cwd(), Path.cwd().parent, checkout)
                  if (p / "src/wisata_madura/protocol.py").is_file()), None)
if repo_root is None:
    if checkout.exists():
        raise FileExistsError("Folder wisata_madura_tuning berisi versi lama. Ganti nama folder atau mulai runtime baru.")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(checkout)], check=True)
    repo_root = checkout
requirements = ["-r", str(repo_root / "requirements-wisata.txt")]
if RUN_DEEP:
    requirements += ["gensim>=4.4,<5", "torch>=2.2,<3"]
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *requirements], check=True)
if any(name.startswith("wisata_madura") for name in sys.modules):
    raise RuntimeError("Modul pernah diimpor. Restart runtime agar kode dan hasil tidak mencampur versi.")
sys.path.insert(0, str(repo_root / "src"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str((OUTPUT_DIR / ".matplotlib").resolve()))
source_commit = subprocess.check_output(["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True).strip()
source_dirty = bool(subprocess.check_output(["git", "-C", str(repo_root), "status", "--porcelain"], text=True).strip())
source_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                 for p in (repo_root / "src/wisata_madura").glob("*.py")}
print("Mode:", MODE, "| Source:", source_commit, "| Output:", OUTPUT_DIR)

## 01 · Input development dan protokol sebelum tuning

Data dimuat otomatis. Rasio fold/seed/kandidat dan objective disimpan sebelum fitting. Jangan mengganti protokol setelah membaca skor untuk mengklaim evaluasi independen.

In [ ]:
import pandas as pd
from IPython.display import display
from wisata_madura.data import load_reviews
if not RUN_DEEP:
    raise ValueError('Tuning memerlukan RUN_DEEP=True dan dependensi deep learning.')
from wisata_madura.tuning import TuningConfig, CANDIDATES, grouped_folds, run_grouped_tuning

manifest=json.loads((repo_root/'data/manifest.json').read_text())
source=repo_root/'data/source'/manifest['source_workbook']
if hashlib.sha256(source.read_bytes()).hexdigest()!=manifest['source_workbook_sha256']:
    raise ValueError('Checksum workbook tidak cocok')
reviews=load_reviews(source)
config=TuningConfig() if MODE=='full' else TuningConfig(training_seeds=(42,), max_epochs=2, patience=1, lda_topics=(2,3), lda_iterations=2)
candidates=CANDIDATES if MODE=='full' else CANDIDATES[:1]
folds,assignment=grouped_folds(reviews,config)
display(pd.DataFrame(candidates))
display(assignment)
for i,(train,valid) in enumerate(folds):
    print(f'Fold {i}: train={len(train)} ulasan/{train.id_destinasi.nunique()} ID; validation={len(valid)} ulasan/{valid.id_destinasi.nunique()} ID')
print('Objective: mean destination macro-F1 kedua target, seluruh fold dan seed.')
print('Kelas singleton train-only; hasil ini bukan final test estimate.')

## 02 · Proses tuning kelompok

Pada setiap fold: LDA dan kosakata fit train; K dipilih pada validation. Panjang urutan ditentukan dari kuantil teks train (dibatasi 16–128 token). FastText fit sekali per kombinasi embedding/panjang/seed lalu dipakai ulang. Dua varian BiLSTM menerima salinan embedding awal; fine-tuning tidak mengubah cache atau inisialisasi ablation berikutnya.

Empat kandidat membandingkan kapasitas, regularisasi, learning rate, dan frozen/fine-tuning. Ini pencarian terbatas, bukan bukti optimum global. Parameter tidak diubah berdasarkan skor setelah pencarian ini. Semua percobaan dan checkpoint dicatat, termasuk yang buruk.

In [ ]:
from dataclasses import asdict
print(asdict(config))
summary,selected=run_grouped_tuning(reviews, OUTPUT_DIR/'search', config, candidates)
display(summary.sort_values('validation_macro_f1_mean',ascending=False))
print(json.dumps(selected,indent=2))

## 03 · Output dan batas kesimpulan

Konfigurasi pemenang hanya terbaik dalam kandidat yang diuji menurut objective validation. Pembanding baseline tetap dicantumkan meski lebih baik daripada kandidat deep. Nilai rerata ± SD fold/seed bukan confidence interval. Kelas train-only tidak punya bukti generalisasi. Checkpoint fold bukan model final refit seluruh data.

Simpan protokol dan jangan menamai skor seleksi ini sebagai independent test. Bandingkan seluruh skor dan cakupan kelas, bukan seed terbaik saja.

In [ ]:
display(pd.read_csv(OUTPUT_DIR/'search/class_coverage.csv'))
display(pd.read_csv(OUTPUT_DIR/'search/checkpoints.csv'))
print('Independent evaluation: PENDING. Deployment ready:',selected['deployment_ready'])
import shutil
archive=shutil.make_archive(str(OUTPUT_DIR),'zip',root_dir=OUTPUT_DIR)
print('Output:',OUTPUT_DIR,'| ZIP:',archive)

## 04 · Evaluasi independen (belum dijalankan)

Kumpulkan ulasan dari destinasi baru setelah protokol/model ditetapkan. Jangan menyalin baris, mengganti ID, atau membagi ulang data lama lalu menyebutnya independen. Simpan asal/tanggal pengumpulan; validator hanya memeriksa overlap, bukan membuktikan provenance independen.

Langkah berikutnya adalah membekukan konfigurasi, menentukan aturan refit/epoch dari development, melatih model final, dan baru membuka label evaluasi eksternal. Belum ada data baru pada eksperimen ini, sehingga tidak dibuat skor independen atau model deployment otomatis.

In [ ]:
# Jalankan hanya setelah tersedia dataset BARU dan protokol model final dibekukan:
# from wisata_madura.tuning import validate_independent_data
# external=load_reviews(Path('/content/data_evaluasi_baru.xlsx'))
# validate_independent_data(reviews,external)
# Lolos pemeriksaan overlap belum berarti evaluasi sudah dilakukan.
print('Tidak ada dataset independen yang digunakan atau dinilai pada notebook tuning ini.')